# Datamine TBLIN Process Example

This notebook demonstrates how to configure and run the **`tblin`** process wrapper in `dmstudio`.

## Process Description

## TBLIN

Table driven "flat-file" data input process.

Enter data of definable format from a system file or the keyboard. The names, types and locations of the input data fields are specified by **EXDEF** , an external definition file. The field layout organization of the input data can be defined with **LAYOUT**. The data may be filtered on input if required. Filtering is optional, and may be specified via fields in **EXDEF** or keywords in **FILTER**.

* **Note** (All filter, end-of-data and page separator matching is case-insensitive. The input data is written to OUT. The data definition is controlled via **PROTODD** , **FIELDLST** and **EXDEF** as follows :):

* If PROTODD is supplied, its data definition is used as a prototype for OUT.

* If FIELDLST is supplied, it specifies a subset of the fields defined in EXDEF for creation in OUT.

* If neither PROTODD nor FIELDLST are supplied, all fields defined in EXDEF are output.

* **Note** (PROTODD AND FIELDLST can not be used at the same time. Data records containing errors may be output to ERROR if desired. All files, EXDEF ,OUT ,ERROR ,FILTER ,PROTODD , FIELDLST if specified, must be different.):

### Input Files:

* **exdef** (Undefined):
  External field definition table. Specifies names, types and locations of input data
  fields. Retrieval criteria, if any, operate on **EXDEF**.
  Required=Yes

* **filter** (Undefined):
  Optional input data filter table.
  Required=No

* **protodd** (Undefined):
  Optional prototype data definition. Selects fields to be created in **OUT**. If not
  supplied, data definition is created for all fields defined in **EXDEF**. Implicit
  fields defined in **EXDEF** are made explicit. Alpha field lengths are taken from

## **EXDEF**.

  Required=No

* **fieldlst** (Undefined):
  Optional field subset list. Selects fields to be created in OUT. If not supplied, data
  definition is define for all fields in EXDEF. One of **PROTODD** , **FIELDLST** can be
  specified. **FIELDLST** must contain at least **FIELD** (A8) Name of field for output.
  (must be subset of **EXDEF**)
  Required=No

### Output Files:

* **out** (Table):
  Output database file to be created.
  Required=Yes

* **error** (Undefined):
  Optional output file for error records. If not supplied, data which is outside
  MIN...MAX limits is placed in OUT.
  Required=No

### Fields:

### Parameters:

* **layout**:
  Input file organisation method. 1=Char Each field located by START, END. 2=Free
  Datamine standard "free" format. 3=Comma Fields separated by commas, no quotes 4=Single
  Fields sep. by commas, quote with ' 5=Double Fields sep. by commas, quote with " 6=White
  Fields separated by spaces/tabs 7=Custom FS and/or DELIM.
  Range=1,7
  Values=1,2,3,4,5,6,7
  Default=1
  Required=Yes

* **delim**:
  Optional field delimiter. Max 4 chars.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **fs**:
  Optional field separator. Max 4 chars.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **skiphd**:
  >=1 Omit n lines of header (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **findpg**:
  >=1 Scan for "page breaks" and omit headers and footers from all pages (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **eod**:
  Optional end of data string. Max 4 char.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **trace**:
  >=1 Display each nth input record (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('tblin')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute tblin
print("Running tblin...")
dm_cmd.tblin(
    exdef_i='t_input_file',  # required
    fieldlst_i='optional',  # required
    out_o='t_tblin_out',  # required
    layout_p='required_val',  # required
    # filter_i='optional',  # optional
    # protodd_i='t_mod1',  # optional
    # error_o='t_tblin_errors',  # optional
    # delim_p='optional',  # optional
    # fs_p='optional',  # optional
    # skiphd_p=0,  # optional
    # findpg_p=0,  # optional
    # eod_p='optional',  # optional
    # trace_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("tblin execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_tblin_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")